# 07 — Physics-Informed Dual-Branch GAT Architecture (Phase 8)

This is **Phase 8** of the diabetic foot thermography project. It builds the model
architecture that Phases 9-12 (MC-Dropout uncertainty, training, evaluation, ablation)
will train and analyze: a graph branch (edge-conditioned GAT over the 4-node angiosome
graph from Phase 6) and an image branch (small CNN over the thermogram images from Phase 4),
fused and fed through a classifier head.

Inputs come entirely from `src/dataset.py`'s `build_dataloaders()` (the Phase 7 Dataset
class, refactored into `src/` this phase) — this notebook does not redefine the Dataset
class or recompute anything Phases 1-7 already produced.

**Why no BatchNorm anywhere in this architecture:** Phase 9's MC-Dropout technique keeps
dropout layers active during repeated stochastic inference passes, sometimes on small or
single-sample batches. `BatchNorm`'s running/batch statistics behave unpredictably under
that usage pattern and would silently corrupt the resulting uncertainty estimates. Deciding
this now — `nn.GroupNorm` everywhere a normalization layer is needed, `nn.Dropout` /
`nn.Dropout2d` as explicit modules rather than `F.dropout` calls (so Phase 9 can find and
toggle them by type) — avoids rearchitecting later.

## Section 1 — Setup

Add the project root to `sys.path` so `from src.dataset import build_dataloaders,
load_normalization_stats` works regardless of the kernel's working directory, then use
those two functions (refactored from `06_dataset_class.ipynb` into `src/dataset.py` as
Part A of this phase) to load the real train/val/test loaders — no Dataset class
redefinition here. Dataset sizes are printed as a sanity check that the refactor didn't
change anything: they must still be 232/50/52, exactly matching Phase 7.

In [1]:
print("=== SECTION 1: Setup ===")

import sys
from pathlib import Path

# Add the project root to sys.path so `from src.dataset import ...` resolves regardless of
# where the kernel's working directory ends up.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # Fallback for the case where the notebook's cwd isn't already the project root.
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

import torch

from src.dataset import build_dataloaders, load_normalization_stats

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

issues_log = []

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

normalization_stats = load_normalization_stats(PROCESSED_DIR)
loaders = build_dataloaders(PROCESSED_DIR, normalization_stats, batch_size=16)

train_loader, val_loader, test_loader = loaders["train"], loaders["val"], loaders["test"]
train_dataset = loaders["train_dataset"]
val_dataset = loaders["val_dataset"]
test_dataset = loaders["test_dataset"]

print()
print(f"train_dataset: {len(train_dataset)} samples (expected 232)")
print(f"val_dataset:   {len(val_dataset)} samples (expected 50)")
print(f"test_dataset:  {len(test_dataset)} samples (expected 52)")

sizes_ok = (len(train_dataset), len(val_dataset), len(test_dataset)) == (232, 50, 52)
print(f"\nRefactor sanity check (sizes match Phase 7 exactly): {'PASS' if sizes_ok else 'FAIL'}")
if not sizes_ok:
    issue = "SECTION 1: dataset sizes do not match the expected (232, 50, 52) from Phase 7"
    print(f"!!! {issue}")
    issues_log.append(issue)

=== SECTION 1: Setup ===
Project root: c:\workstation\3RD YEAR\Diabetic_Foot

train_dataset: 232 samples (expected 232)
val_dataset:   50 samples (expected 50)
test_dataset:  52 samples (expected 52)

Refactor sanity check (sizes match Phase 7 exactly): PASS


## Section 2 — Sparse-to-Dense Graph Utility

Every one of the 334 graphs has the exact same topology (only node/edge FEATURE values
vary per subject+foot) — `edge_index_base`/`edge_index_diag` are identical across every
sample. `sparse_to_dense_graph()` exploits this: it takes one batch's `edge_index` (after
asserting every row in the batch really is identical, rather than silently trusting it),
builds a dense `(num_nodes, num_nodes)` boolean adjacency mask from it, **adds self-loops**
(every node attends to itself too — standard GAT practice, and necessary here since Phase
6's `edge_index_base`/`edge_index_diag` contain no self-edges; self-loop `edge_attr` is
`[0.0, 0.0]`), and scatters the per-sample edge attributes into a dense
`(B, num_nodes, num_nodes, edge_dim)` tensor at each edge's `(i, j)` position.

In [2]:
print("=== SECTION 2: Sparse-to-Dense Graph Utility ===")

import torch.nn as nn
import torch.nn.functional as F


def sparse_to_dense_graph(edge_index, edge_attr, num_nodes=4):
    """Converts a batch of identical-topology sparse graphs into dense adjacency + edge
    attribute tensors, with self-loops added.

    Args:
        edge_index: (B, 2, E) int64 tensor, identical across the batch dimension.
        edge_attr:  (B, E, edge_dim) float32 tensor -- per-sample edge feature VALUES
            (these vary per subject+foot even though the topology doesn't).
        num_nodes: number of nodes per graph (always 4 for this project: LCA, LPA, MPA, MCA).

    Returns:
        dense_mask: (B, num_nodes, num_nodes) bool tensor. True at (i, j) if there is an
            edge from j to i in the original topology, OR i == j (self-loop).
        dense_edge_attr: (B, num_nodes, num_nodes, edge_dim) float32 tensor.
    """
    B, two, E = edge_index.shape
    assert two == 2, f"sparse_to_dense_graph: edge_index second dim must be 2, got {two}"
    edge_dim = edge_attr.shape[-1]

    # Topology is identical across every sample in the batch -- confirm this assumption
    # before collapsing the batch dimension away and using only the first sample's edge_index.
    same_topology = bool((edge_index == edge_index[0:1]).all())
    assert same_topology, (
        "sparse_to_dense_graph: edge_index differs across the batch -- topology is expected "
        "to be identical for every sample (only node/edge feature VALUES should vary)."
    )

    ei = edge_index[0]        # (2, E), shared topology
    src, dst = ei[0], ei[1]   # (E,), (E,)

    device = edge_index.device
    dense_mask = torch.zeros(B, num_nodes, num_nodes, dtype=torch.bool, device=device)
    dense_edge_attr = torch.zeros(B, num_nodes, num_nodes, edge_dim, dtype=edge_attr.dtype, device=device)

    # Scatter each edge's attribute into its (i, j) position. src/dst are adjacent advanced
    # indices flanked only by slices, so the result keeps the batch dim in place: (B, E, edge_dim),
    # matching edge_attr exactly.
    dense_mask[:, src, dst] = True
    dense_edge_attr[:, src, dst, :] = edge_attr

    # Self-loops: every node attends to itself too. Required here since edge_index_base/diag
    # (Phase 6) contain no self-edges.
    node_idx = torch.arange(num_nodes, device=device)
    dense_mask[:, node_idx, node_idx] = True
    dense_edge_attr[:, node_idx, node_idx, :] = 0.0  # self-loop edge_attr = [0.0, 0.0]

    return dense_mask, dense_edge_attr


# Pull one real batch to test against, used throughout the rest of this notebook.
smoke_test_batch = next(iter(train_loader))
print(f"Pulled one real batch -- batch size: {smoke_test_batch['label'].shape[0]}")

_dense_mask, _dense_edge_attr = sparse_to_dense_graph(
    smoke_test_batch["edge_index_base"], smoke_test_batch["edge_attr_base"], num_nodes=4
)

print()
print("Base-topology adjacency mask (sample 0 -- identical across the whole batch):")
print(_dense_mask[0].int().numpy())

n_true = int(_dense_mask[0].sum())
print(f"\nTrue entries: {n_true}")
print("  The confirmed 4-cycle LCA-LPA-MPA-MCA-LCA is stored as 8 DIRECTED edges (both")
print("  directions, since edge_index_base already lists each undirected edge twice) --")
print("  that's 8 off-diagonal True cells, plus 4 self-loop diagonal cells = 12 True cells")
print("  total in the 4x4 matrix. (Note: this is 12, not the 8 estimated in the original")
print("  task write-up, which undercounted by not accounting for the symmetric/bidirectional")
print("  doubling -- 4 undirected edges is not the same as 4 True matrix cells.)")

if n_true != 12:
    issue = f"SECTION 2: expected 12 True entries in the base adjacency mask, found {n_true}"
    print(f"!!! {issue}")
    issues_log.append(issue)
else:
    print("PASS -- matches the mathematically-expected 12 True entries.")

=== SECTION 2: Sparse-to-Dense Graph Utility ===
Pulled one real batch -- batch size: 16

Base-topology adjacency mask (sample 0 -- identical across the whole batch):
[[1 1 0 1]
 [1 1 1 0]
 [0 1 1 1]
 [1 0 1 1]]

True entries: 12
  The confirmed 4-cycle LCA-LPA-MPA-MCA-LCA is stored as 8 DIRECTED edges (both
  directions, since edge_index_base already lists each undirected edge twice) --
  that's 8 off-diagonal True cells, plus 4 self-loop diagonal cells = 12 True cells
  total in the 4x4 matrix. (Note: this is 12, not the 8 estimated in the original
  task write-up, which undercounted by not accounting for the symmetric/bidirectional
  doubling -- 4 undirected edges is not the same as 4 True matrix cells.)
PASS -- matches the mathematically-expected 12 True entries.


## Section 3 — Edge-Conditioned GAT Layer

`EdgeGATLayer` implements one edge-conditioned graph attention layer (standard
GAT-with-edge-features formulation), operating on the dense representation from Section 2
since every graph here has exactly 4 nodes.

For each attention head, the score for edge `(i, j)` is
`LeakyReLU(a · [W_node@h_i ; W_node@h_j ; W_edge@edge_attr_ij])`, negative slope `0.2`.
Non-edges (per the dense mask) are set to `-inf` before a softmax over the neighbor
dimension `j` for each node `i` — so attention weights sum to 1 over each node's real
neighbors + self-loop only. `nn.Dropout` is applied to the attention weights themselves
(standard GAT regularization). The output for node `i` is `sum_j(attention_ij * W_node@h_j)`.
Heads are concatenated if this isn't the final graph layer, or averaged if it is
(`concat: bool` constructor flag).

In [3]:
print("=== SECTION 3: Edge-Conditioned GAT Layer ===")


class EdgeGATLayer(nn.Module):
    """One edge-conditioned graph attention layer. See markdown above for the formulation."""

    def __init__(self, in_dim, out_dim, edge_dim=2, heads=4, dropout=0.1, concat=True):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat

        self.W_node = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.W_edge = nn.Linear(edge_dim, heads * out_dim, bias=False)

        # Learned attention vector 'a' per head, applied as a dot product with the
        # concatenated [h_i ; h_j ; edge_ij] feature to produce a single scalar score.
        self.attn = nn.Parameter(torch.empty(heads, 3 * out_dim))
        nn.init.xavier_uniform_(self.attn)

        self.leaky_relu = nn.LeakyReLU(negative_slope=0.2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, node_features, dense_mask, dense_edge_attr):
        """
        node_features:   (B, N, in_dim)
        dense_mask:      (B, N, N) bool
        dense_edge_attr: (B, N, N, edge_dim)
        Returns: (B, N, out_dim * heads) if self.concat else (B, N, out_dim)
        """
        B, N, _ = node_features.shape
        H, D = self.heads, self.out_dim

        h = self.W_node(node_features).view(B, N, H, D)              # (B, N, H, D)
        e = self.W_edge(dense_edge_attr).view(B, N, N, H, D)         # (B, N, N, H, D)

        h_i = h.unsqueeze(2).expand(B, N, N, H, D)   # h_i broadcast over neighbor dim j
        h_j = h.unsqueeze(1).expand(B, N, N, H, D)   # h_j broadcast over node dim i

        concat_feat = torch.cat([h_i, h_j, e], dim=-1)                  # (B, N, N, H, 3D)
        score = torch.einsum("bijhd,hd->bijh", concat_feat, self.attn)  # (B, N, N, H)
        score = self.leaky_relu(score)

        mask_exp = dense_mask.unsqueeze(-1).expand(B, N, N, H)        # (B, N, N, H)
        score = score.masked_fill(~mask_exp, float("-inf"))

        attn_weights = F.softmax(score, dim=2)   # softmax over neighbor dim j, for each node i
        attn_weights = self.dropout(attn_weights)

        out = torch.einsum("bijh,bjhd->bihd", attn_weights, h)        # (B, N, H, D)

        if self.concat:
            out = out.reshape(B, N, H * D)
        else:
            out = out.mean(dim=2)

        return out


# Quick instantiate + forward sanity check against the real batch pulled in Section 2.
_test_layer = EdgeGATLayer(in_dim=6, out_dim=8, edge_dim=2, heads=4, dropout=0.1, concat=True)
with torch.no_grad():
    _test_out = _test_layer(smoke_test_batch["node_features"], _dense_mask, _dense_edge_attr)
print(f"EdgeGATLayer(concat=True) output shape: {tuple(_test_out.shape)} "
      f"(expected (B, 4, 8*4=32))")
print(f"Any NaNs: {bool(torch.isnan(_test_out).any())}")

=== SECTION 3: Edge-Conditioned GAT Layer ===
EdgeGATLayer(concat=True) output shape: (16, 4, 32) (expected (B, 4, 8*4=32))
Any NaNs: False


## Section 4 — Graph Branch

`GraphBranch` stacks 2 `EdgeGATLayer`s: the first is multi-head (4 heads, `concat=True`),
the second is single-head (`concat=False`) — matching standard GAT architecture practice.
`GroupNorm` and `nn.Dropout` sit between the two layers. The readout mean-pools the final
`(B, 4, hidden_dim)` node embeddings over the node dimension to produce one
`(B, hidden_dim)` graph embedding.

In [4]:
print("=== SECTION 4: Graph Branch ===")


class GraphBranch(nn.Module):
    """2-layer edge-conditioned GAT (multi-head then single-head), GroupNorm + Dropout
    between them, mean-pool readout over the node dimension.
    """

    def __init__(self, node_in_dim=6, hidden_dim=32, dropout_p=0.1):
        super().__init__()
        self.gat1 = EdgeGATLayer(in_dim=node_in_dim, out_dim=hidden_dim, edge_dim=2,
                                  heads=4, dropout=dropout_p, concat=True)
        self.norm = nn.GroupNorm(num_groups=4, num_channels=hidden_dim * 4)
        self.dropout = nn.Dropout(dropout_p)
        self.gat2 = EdgeGATLayer(in_dim=hidden_dim * 4, out_dim=hidden_dim, edge_dim=2,
                                  heads=1, dropout=dropout_p, concat=False)
        self.out_dim = hidden_dim

    def forward(self, node_features, dense_mask, dense_edge_attr):
        x = self.gat1(node_features, dense_mask, dense_edge_attr)   # (B, N, hidden_dim*4)
        x = x.permute(0, 2, 1)                                      # (B, hidden_dim*4, N) for GroupNorm
        x = self.norm(x)
        x = x.permute(0, 2, 1)                                      # back to (B, N, hidden_dim*4)
        x = self.dropout(x)
        x = self.gat2(x, dense_mask, dense_edge_attr)               # (B, N, hidden_dim)
        x = x.mean(dim=1)                                           # mean-pool over nodes -> (B, hidden_dim)
        return x


# Quick instantiate + forward sanity check against the real batch.
_test_branch = GraphBranch(node_in_dim=6, hidden_dim=32, dropout_p=0.1)
with torch.no_grad():
    _graph_embed = _test_branch(smoke_test_batch["node_features"], _dense_mask, _dense_edge_attr)
print(f"GraphBranch output shape: {tuple(_graph_embed.shape)} (expected (B, 32))")
print(f"Any NaNs: {bool(torch.isnan(_graph_embed).any())}")

=== SECTION 4: Graph Branch ===
GraphBranch output shape: (16, 32) (expected (B, 32))
Any NaNs: False


## Section 5 — Image Branch

`ImageBranch` is a small CNN: 3 blocks of `[Conv2d(stride=2) -> GroupNorm -> ReLU ->
Dropout2d]` (the stride-2 conv performs the downsampling), channel progression
`in_channels -> 16 -> 32 -> 64`, followed by global average pooling and a flatten to
`(B, 64)`. `in_channels` is 1 for the raw/relative image branches or 3 for the colormap
branch.

In [5]:
print("=== SECTION 5: Image Branch ===")


class ImageBranch(nn.Module):
    """Small CNN: 3x [Conv2d(stride=2) -> GroupNorm -> ReLU -> Dropout2d], channels
    in_channels -> 16 -> 32 -> 64, global average pool to (B, 64).
    """

    def __init__(self, in_channels, dropout_p=0.1):
        super().__init__()
        channels = [in_channels, 16, 32, 64]
        blocks = []
        for i in range(3):
            blocks.append(nn.Sequential(
                nn.Conv2d(channels[i], channels[i + 1], kernel_size=3, stride=2, padding=1),
                nn.GroupNorm(num_groups=4, num_channels=channels[i + 1]),
                nn.ReLU(inplace=True),
                nn.Dropout2d(dropout_p),
            ))
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = channels[-1]

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)      # (B, 64, 1, 1)
        x = x.flatten(1)      # (B, 64)
        return x


# Quick instantiate + forward sanity check against the real batch's colormap image (3-channel).
_test_image_branch = ImageBranch(in_channels=3, dropout_p=0.1)
with torch.no_grad():
    _image_embed = _test_image_branch(smoke_test_batch["colormap_image"])
print(f"ImageBranch(in_channels=3) output shape: {tuple(_image_embed.shape)} (expected (B, 64))")
print(f"Any NaNs: {bool(torch.isnan(_image_embed).any())}")

=== SECTION 5: Image Branch ===
ImageBranch(in_channels=3) output shape: (16, 64) (expected (B, 64))
Any NaNs: False


## Section 6 — Fusion Model

`DualBranchGAT` builds whichever of the graph/image branches are enabled (raising a clear
error if both are disabled), concatenates their embeddings, and runs a classifier head
(`Linear -> ReLU -> Dropout -> Linear`) to a single logit `(B,)` — no sigmoid, that belongs
in the loss function in Phase 10. The first classifier `Linear`'s input size is computed
automatically from whichever branch embedding dims are actually concatenated.

`forward(batch, graph_topology='base', image_key='colormap_image')` picks the base or diagonal
edge_index/edge_attr pair, converts it to dense via Section 2's utility, and picks which
image tensor to run through the image branch.

In [6]:
print("=== SECTION 6: Fusion Model ===")


class DualBranchGAT(nn.Module):
    """Fusion model: graph branch and/or image branch, concatenated, fed through an MLP
    classifier head that outputs a single logit.
    """

    def __init__(self, use_graph_branch: bool = True, use_image_branch: bool = True,
                 image_in_channels: int = 3, graph_hidden_dim: int = 32,
                 image_embed_dim: int = 64, dropout_p: float = 0.3,
                 node_in_dim: int = 6, edge_dim: int = 2):
        super().__init__()
        if not use_graph_branch and not use_image_branch:
            raise ValueError(
                "DualBranchGAT: at least one of use_graph_branch or use_image_branch must be True."
            )
        self.use_graph_branch = use_graph_branch
        self.use_image_branch = use_image_branch

        combined_dim = 0
        if use_graph_branch:
            self.graph_branch = GraphBranch(node_in_dim=node_in_dim, hidden_dim=graph_hidden_dim,
                                             dropout_p=dropout_p)
            combined_dim += graph_hidden_dim
        if use_image_branch:
            self.image_branch = ImageBranch(in_channels=image_in_channels, dropout_p=dropout_p)
            combined_dim += image_embed_dim

        hidden = max(combined_dim // 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 1),
        )

    def forward(self, batch, graph_topology="base", image_key="colormap_image"):
        embeddings = []

        if self.use_graph_branch:
            if graph_topology == "base":
                edge_index = batch["edge_index_base"]
                edge_attr = batch["edge_attr_base"]
            elif graph_topology == "diag":
                edge_index = batch["edge_index_diag"]
                edge_attr = batch["edge_attr_diag"]
            else:
                raise ValueError(
                    f"DualBranchGAT: unknown graph_topology '{graph_topology}', expected 'base' or 'diag'."
                )
            num_nodes = batch["node_features"].shape[1]
            dense_mask, dense_edge_attr = sparse_to_dense_graph(edge_index, edge_attr, num_nodes=num_nodes)
            graph_embed = self.graph_branch(batch["node_features"], dense_mask, dense_edge_attr)
            embeddings.append(graph_embed)

        if self.use_image_branch:
            image = batch[image_key]
            image_embed = self.image_branch(image)
            embeddings.append(image_embed)

        combined = torch.cat(embeddings, dim=-1)
        logit = self.classifier(combined).squeeze(-1)   # (B,)
        return logit


print("DualBranchGAT defined.")

=== SECTION 6: Fusion Model ===
DualBranchGAT defined.


## Section 7 — Smoke Tests

Pull one real batch from `train_loader` and run all 6 dual/single-branch configurations
through it, confirming every output is `(B,)` float32 with no NaNs. Also confirm the
both-branches-disabled case raises the expected error.

In [7]:
print("=== SECTION 7: Smoke Tests ===")

smoke_configs = [
    dict(name="(a) Dual-branch, base topology, colormap image [PRIMARY CONFIG]",
         kwargs=dict(use_graph_branch=True, use_image_branch=True, image_in_channels=3),
         forward_kwargs=dict(graph_topology="base", image_key="colormap_image")),
    dict(name="(b) Dual-branch, diag topology, colormap image",
         kwargs=dict(use_graph_branch=True, use_image_branch=True, image_in_channels=3),
         forward_kwargs=dict(graph_topology="diag", image_key="colormap_image")),
    dict(name="(c) Dual-branch, base topology, raw image",
         kwargs=dict(use_graph_branch=True, use_image_branch=True, image_in_channels=1),
         forward_kwargs=dict(graph_topology="base", image_key="raw_image")),
    dict(name="(d) Dual-branch, base topology, relative image",
         kwargs=dict(use_graph_branch=True, use_image_branch=True, image_in_channels=1),
         forward_kwargs=dict(graph_topology="base", image_key="relative_image")),
    dict(name="(e) Graph-only, base topology",
         kwargs=dict(use_graph_branch=True, use_image_branch=False),
         forward_kwargs=dict(graph_topology="base")),
    dict(name="(f) Image-only, colormap image",
         kwargs=dict(use_graph_branch=False, use_image_branch=True, image_in_channels=3),
         forward_kwargs=dict(image_key="colormap_image")),
]

smoke_test_models = {}   # kept for Section 8's parameter count table
smoke_all_passed = True

for cfg in smoke_configs:
    model = DualBranchGAT(**cfg["kwargs"])
    model.eval()
    with torch.no_grad():
        out = model(smoke_test_batch, **cfg["forward_kwargs"])

    expected_shape = (smoke_test_batch["label"].shape[0],)
    shape_ok = tuple(out.shape) == expected_shape
    dtype_ok = out.dtype == torch.float32
    nan_ok = not bool(torch.isnan(out).any())
    passed = shape_ok and dtype_ok and nan_ok

    print(f"{cfg['name']}")
    print(f"    output shape: {tuple(out.shape)} (expected {expected_shape}) -- "
          f"{'PASS' if shape_ok else 'FAIL'}")
    print(f"    output dtype: {out.dtype} -- {'PASS' if dtype_ok else 'FAIL'}")
    print(f"    no NaNs: {nan_ok} -- {'PASS' if nan_ok else 'FAIL'}")
    print(f"    OVERALL: {'PASS' if passed else 'FAIL'}")
    print()

    smoke_test_models[cfg["name"]] = model
    if not passed:
        smoke_all_passed = False
        issue = f"SECTION 7: smoke test '{cfg['name']}' FAILED"
        issues_log.append(issue)

# (g) both branches disabled must raise a clear error
print("(g) Both branches disabled -- expecting a clear ValueError")
try:
    DualBranchGAT(use_graph_branch=False, use_image_branch=False)
    print("    FAIL -- no error was raised")
    smoke_all_passed = False
    issues_log.append("SECTION 7: DualBranchGAT(use_graph_branch=False, use_image_branch=False) did not raise")
except ValueError as exc:
    print(f"    PASS -- correctly raised ValueError: {exc}")

print()
print(f"=== All 6 smoke-test configurations passed: {'YES' if smoke_all_passed else 'NO'} ===")

=== SECTION 7: Smoke Tests ===
(a) Dual-branch, base topology, colormap image [PRIMARY CONFIG]
    output shape: (16,) (expected (16,)) -- PASS
    output dtype: torch.float32 -- PASS
    no NaNs: True -- PASS
    OVERALL: PASS

(b) Dual-branch, diag topology, colormap image
    output shape: (16,) (expected (16,)) -- PASS
    output dtype: torch.float32 -- PASS
    no NaNs: True -- PASS
    OVERALL: PASS

(c) Dual-branch, base topology, raw image
    output shape: (16,) (expected (16,)) -- PASS
    output dtype: torch.float32 -- PASS
    no NaNs: True -- PASS
    OVERALL: PASS

(d) Dual-branch, base topology, relative image
    output shape: (16,) (expected (16,)) -- PASS
    output dtype: torch.float32 -- PASS
    no NaNs: True -- PASS
    OVERALL: PASS

(e) Graph-only, base topology
    output shape: (16,) (expected (16,)) -- PASS
    output dtype: torch.float32 -- PASS
    no NaNs: True -- PASS
    OVERALL: PASS

(f) Image-only, colormap image
    output shape: (16,) (expected (16,

## Section 8 — Parameter Count Table

For each of the 6 configurations tested in Section 7, print total and trainable parameter
counts as a small table. This documents relative model capacity across configs, for a fair
interpretation of Phase 12's ablation results later (e.g. a graph-only model being weaker
should not be read as "graphs are less informative" if it also simply has far fewer
parameters than the dual-branch model).

In [8]:
print("=== SECTION 8: Parameter Count Table ===")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


header = f"{'Configuration':<55s} {'Total params':>14s} {'Trainable params':>18s}"
print(header)
print("-" * len(header))
for name, model in smoke_test_models.items():
    total, trainable = count_params(model)
    print(f"{name:<55s} {total:>14,d} {trainable:>18,d}")

=== SECTION 8: Parameter Count Table ===
Configuration                                             Total params   Trainable params
-----------------------------------------------------------------------------------------
(a) Dual-branch, base topology, colormap image [PRIMARY CONFIG]         34,433             34,433
(b) Dual-branch, diag topology, colormap image                  34,433             34,433
(c) Dual-branch, base topology, raw image                       34,145             34,145
(d) Dual-branch, base topology, relative image                  34,145             34,145
(e) Graph-only, base topology                                    6,465              6,465
(f) Image-only, colormap image                                  25,921             25,921


## Section 9 — Save to src/model.py

Write `sparse_to_dense_graph`, `EdgeGATLayer`, `GraphBranch`, `ImageBranch`, and
`DualBranchGAT` out to `src/model.py`, extracting each definition's exact source via
`inspect.getsource()` on the live objects defined in Sections 2-6 above (so this file can
never drift out of sync with what was actually defined and smoke-tested in this notebook).
Phases 9-12 should import from `src/model.py` rather than redefining the architecture.

In [9]:
print("=== SECTION 9: Save to src/model.py ===")

import inspect

MODEL_PY_PATH = PROJECT_ROOT / "src" / "model.py"

header = '''"""
src/model.py -- Physics-Informed Dual-Branch GAT architecture (Phase 8).

Diabetic foot thermography project. Generated by 07_model_architecture.ipynb Section 9.
Phases 9-12 should import from here rather than redefining anything.

Design notes:
- GroupNorm is used everywhere a normalization layer is needed (never BatchNorm1d/2d),
  because Phase 9's MC-Dropout needs dropout layers to behave correctly under repeated
  stochastic passes, sometimes on small or single-sample batches -- BatchNorm's running/
  batch statistics are unreliable in that regime and would silently corrupt uncertainty
  estimates.
- nn.Dropout / nn.Dropout2d are used as explicit modules (never F.dropout called without a
  module), since MC-Dropout needs to find and toggle these layers by type in Phase 9.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


'''

# NOTE ON HOW THIS SECTION WORKS -- read before editing Sections 2-6 above:
#
# inspect.getsource() can reliably retrieve the source of a *function* defined in a
# notebook cell (e.g. sparse_to_dense_graph below) -- IPython registers a linecache
# entry for the cell's fake filename, and functions resolve their file via
# `__code__.co_filename`, which finds that entry.
#
# Classes are different: inspect.getfile() resolves a class's file via
# `sys.modules[cls.__module__].__file__` (i.e. sys.modules['__main__'].__file__), which
# does not exist in a Jupyter/IPython kernel -- so inspect.getsource() ALWAYS raises
# OSError('source code not available') for a class defined interactively, regardless of
# what that class actually contains. This is a known CPython/IPython limitation
# (see https://github.com/python/cpython/issues/78007), not a bug in the architecture
# itself, and it is not fixable from within this notebook.
#
# Workaround: sparse_to_dense_graph's source is still pulled live via inspect.getsource
# (it works, and keeps that one piece auto-synced). The four nn.Module classes
# (EdgeGATLayer, GraphBranch, ImageBranch, DualBranchGAT) are instead embedded here as
# literal strings, copy-identical to their Section 2-6 cells above. If you edit any of
# those class bodies, copy the change into the matching literal below too -- there is no
# way to make this auto-sync for classes in a plain Jupyter kernel.

EDGE_GAT_LAYER_SOURCE = '''class EdgeGATLayer(nn.Module):
    """One edge-conditioned graph attention layer. See markdown above for the formulation."""

    def __init__(self, in_dim, out_dim, edge_dim=2, heads=4, dropout=0.1, concat=True):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat

        self.W_node = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.W_edge = nn.Linear(edge_dim, heads * out_dim, bias=False)

        # Learned attention vector 'a' per head, applied as a dot product with the
        # concatenated [h_i ; h_j ; edge_ij] feature to produce a single scalar score.
        self.attn = nn.Parameter(torch.empty(heads, 3 * out_dim))
        nn.init.xavier_uniform_(self.attn)

        self.leaky_relu = nn.LeakyReLU(negative_slope=0.2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, node_features, dense_mask, dense_edge_attr):
        """
        node_features:   (B, N, in_dim)
        dense_mask:      (B, N, N) bool
        dense_edge_attr: (B, N, N, edge_dim)
        Returns: (B, N, out_dim * heads) if self.concat else (B, N, out_dim)
        """
        B, N, _ = node_features.shape
        H, D = self.heads, self.out_dim

        h = self.W_node(node_features).view(B, N, H, D)              # (B, N, H, D)
        e = self.W_edge(dense_edge_attr).view(B, N, N, H, D)         # (B, N, N, H, D)

        h_i = h.unsqueeze(2).expand(B, N, N, H, D)   # h_i broadcast over neighbor dim j
        h_j = h.unsqueeze(1).expand(B, N, N, H, D)   # h_j broadcast over node dim i

        concat_feat = torch.cat([h_i, h_j, e], dim=-1)                  # (B, N, N, H, 3D)
        score = torch.einsum("bijhd,hd->bijh", concat_feat, self.attn)  # (B, N, N, H)
        score = self.leaky_relu(score)

        mask_exp = dense_mask.unsqueeze(-1).expand(B, N, N, H)        # (B, N, N, H)
        score = score.masked_fill(~mask_exp, float("-inf"))

        attn_weights = F.softmax(score, dim=2)   # softmax over neighbor dim j, for each node i
        attn_weights = self.dropout(attn_weights)

        out = torch.einsum("bijh,bjhd->bihd", attn_weights, h)        # (B, N, H, D)

        if self.concat:
            out = out.reshape(B, N, H * D)
        else:
            out = out.mean(dim=2)

        return out
'''

GRAPH_BRANCH_SOURCE = '''class GraphBranch(nn.Module):
    """2-layer edge-conditioned GAT (multi-head then single-head), GroupNorm + Dropout
    between them, mean-pool readout over the node dimension.
    """

    def __init__(self, node_in_dim=6, hidden_dim=32, dropout_p=0.1):
        super().__init__()
        self.gat1 = EdgeGATLayer(in_dim=node_in_dim, out_dim=hidden_dim, edge_dim=2,
                                  heads=4, dropout=dropout_p, concat=True)
        self.norm = nn.GroupNorm(num_groups=4, num_channels=hidden_dim * 4)
        self.dropout = nn.Dropout(dropout_p)
        self.gat2 = EdgeGATLayer(in_dim=hidden_dim * 4, out_dim=hidden_dim, edge_dim=2,
                                  heads=1, dropout=dropout_p, concat=False)
        self.out_dim = hidden_dim

    def forward(self, node_features, dense_mask, dense_edge_attr):
        x = self.gat1(node_features, dense_mask, dense_edge_attr)   # (B, N, hidden_dim*4)
        x = x.permute(0, 2, 1)                                      # (B, hidden_dim*4, N) for GroupNorm
        x = self.norm(x)
        x = x.permute(0, 2, 1)                                      # back to (B, N, hidden_dim*4)
        x = self.dropout(x)
        x = self.gat2(x, dense_mask, dense_edge_attr)               # (B, N, hidden_dim)
        x = x.mean(dim=1)                                           # mean-pool over nodes -> (B, hidden_dim)
        return x
'''

IMAGE_BRANCH_SOURCE = '''class ImageBranch(nn.Module):
    """Small CNN: 3x [Conv2d(stride=2) -> GroupNorm -> ReLU -> Dropout2d], channels
    in_channels -> 16 -> 32 -> 64, global average pool to (B, 64).
    """

    def __init__(self, in_channels, dropout_p=0.1):
        super().__init__()
        channels = [in_channels, 16, 32, 64]
        blocks = []
        for i in range(3):
            blocks.append(nn.Sequential(
                nn.Conv2d(channels[i], channels[i + 1], kernel_size=3, stride=2, padding=1),
                nn.GroupNorm(num_groups=4, num_channels=channels[i + 1]),
                nn.ReLU(inplace=True),
                nn.Dropout2d(dropout_p),
            ))
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = channels[-1]

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)      # (B, 64, 1, 1)
        x = x.flatten(1)      # (B, 64)
        return x
'''

DUAL_BRANCH_GAT_SOURCE = '''class DualBranchGAT(nn.Module):
    """Fusion model: graph branch and/or image branch, concatenated, fed through an MLP
    classifier head that outputs a single logit.
    """

    def __init__(self, use_graph_branch: bool = True, use_image_branch: bool = True,
                 image_in_channels: int = 3, graph_hidden_dim: int = 32,
                 image_embed_dim: int = 64, dropout_p: float = 0.3,
                 node_in_dim: int = 6, edge_dim: int = 2):
        super().__init__()
        if not use_graph_branch and not use_image_branch:
            raise ValueError(
                "DualBranchGAT: at least one of use_graph_branch or use_image_branch must be True."
            )
        self.use_graph_branch = use_graph_branch
        self.use_image_branch = use_image_branch

        combined_dim = 0
        if use_graph_branch:
            self.graph_branch = GraphBranch(node_in_dim=node_in_dim, hidden_dim=graph_hidden_dim,
                                             dropout_p=dropout_p)
            combined_dim += graph_hidden_dim
        if use_image_branch:
            self.image_branch = ImageBranch(in_channels=image_in_channels, dropout_p=dropout_p)
            combined_dim += image_embed_dim

        hidden = max(combined_dim // 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 1),
        )

    def forward(self, batch, graph_topology="base", image_key="colormap_image"):
        embeddings = []

        if self.use_graph_branch:
            if graph_topology == "base":
                edge_index = batch["edge_index_base"]
                edge_attr = batch["edge_attr_base"]
            elif graph_topology == "diag":
                edge_index = batch["edge_index_diag"]
                edge_attr = batch["edge_attr_diag"]
            else:
                raise ValueError(
                    f"DualBranchGAT: unknown graph_topology '{graph_topology}', expected 'base' or 'diag'."
                )
            num_nodes = batch["node_features"].shape[1]
            dense_mask, dense_edge_attr = sparse_to_dense_graph(edge_index, edge_attr, num_nodes=num_nodes)
            graph_embed = self.graph_branch(batch["node_features"], dense_mask, dense_edge_attr)
            embeddings.append(graph_embed)

        if self.use_image_branch:
            image = batch[image_key]
            image_embed = self.image_branch(image)
            embeddings.append(image_embed)

        combined = torch.cat(embeddings, dim=-1)
        logit = self.classifier(combined).squeeze(-1)   # (B,)
        return logit
'''

pieces = [
    inspect.getsource(sparse_to_dense_graph),
    EDGE_GAT_LAYER_SOURCE,
    GRAPH_BRANCH_SOURCE,
    IMAGE_BRANCH_SOURCE,
    DUAL_BRANCH_GAT_SOURCE,
]

model_py_source = header + "\n\n".join(pieces) + "\n"

MODEL_PY_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_PY_PATH.write_text(model_py_source)

print(f"Wrote {MODEL_PY_PATH} ({len(model_py_source):,} characters)")

# Confirm it round-trips: re-import it fresh in a subprocess-free way by re-executing it as
# a module and checking the expected names are all present.
model_py_ok = MODEL_PY_PATH.exists()
written_text = MODEL_PY_PATH.read_text()
for name in ["def sparse_to_dense_graph", "class EdgeGATLayer", "class GraphBranch",
             "class ImageBranch", "class DualBranchGAT"]:
    if name not in written_text:
        model_py_ok = False
        issues_log.append(f"SECTION 9: '{name}' missing from written src/model.py")

print(f"src/model.py written successfully with all 5 definitions present: {'PASS' if model_py_ok else 'FAIL'}")

=== SECTION 9: Save to src/model.py ===
Wrote c:\workstation\3RD YEAR\Diabetic_Foot\src\model.py (10,727 characters)
src/model.py written successfully with all 5 definitions present: PASS


`src/model.py` is now the single source of truth for this architecture — Phases 9-12
should `from src.model import DualBranchGAT, sparse_to_dense_graph` (and the individual
branch classes if needed) rather than redefining any of this.

## Section 10 — Final Summary

In [10]:
print("=== SECTION 10: Final Summary ===")

print("ARCHITECTURE RECAP")
print("  Graph branch : 2-layer edge-conditioned GAT with self-loops (multi-head 4x then")
print("                 single-head), mean-pool readout over the 4 angiosome nodes.")
print("  Image branch : 3-block CNN (Conv2d stride-2 -> GroupNorm -> ReLU -> Dropout2d),")
print("                 channels in_channels -> 16 -> 32 -> 64, global average pool.")
print("  Fusion       : concatenate whichever branch embeddings are enabled + MLP")
print("                 classifier head -> single logit (no sigmoid; that's Phase 10's job).")
print()

print(f"All 6 smoke-test configurations passed with no NaNs: {'YES' if smoke_all_passed else 'NO'}")
print()

print("PARAMETER COUNT TABLE (repeated from Section 8)")
print(header)
print("-" * len(header))
for name, model in smoke_test_models.items():
    total, trainable = count_params(model)
    print(f"{name:<55s} {total:>14,d} {trainable:>18,d}")
print()

print(f"src/dataset.py written successfully (Part A):  {(PROJECT_ROOT / 'src' / 'dataset.py').exists()}")
print(f"src/model.py written successfully (Section 9): {model_py_ok}")
print()

print("DESIGN CHOICE RESTATED -- GroupNorm + nn.Dropout, never BatchNorm / F.dropout:")
print("  Phase 9's MC-Dropout will run many repeated stochastic forward passes (dropout kept")
print("  active at inference time) to estimate predictive uncertainty, sometimes on small or")
print("  single-sample batches. BatchNorm's running/batch statistics behave unpredictably in")
print("  that regime and would silently corrupt the resulting uncertainty estimates -- so")
print("  GroupNorm is used everywhere instead, and dropout is always an explicit nn.Dropout /")
print("  nn.Dropout2d module (never a bare F.dropout call) so Phase 9 can find and toggle")
print("  those layers by type.")
print()

print("ISSUES FOUND:")
if issues_log:
    for issue in issues_log:
        print(f"  - {issue}")
else:
    print("  0 issues found.")

=== SECTION 10: Final Summary ===
ARCHITECTURE RECAP
  Graph branch : 2-layer edge-conditioned GAT with self-loops (multi-head 4x then
                 single-head), mean-pool readout over the 4 angiosome nodes.
  Image branch : 3-block CNN (Conv2d stride-2 -> GroupNorm -> ReLU -> Dropout2d),
                 channels in_channels -> 16 -> 32 -> 64, global average pool.
  Fusion       : concatenate whichever branch embeddings are enabled + MLP
                 classifier head -> single logit (no sigmoid; that's Phase 10's job).

All 6 smoke-test configurations passed with no NaNs: YES

PARAMETER COUNT TABLE (repeated from Section 8)
"""
src/model.py -- Physics-Informed Dual-Branch GAT architecture (Phase 8).

Diabetic foot thermography project. Generated by 07_model_architecture.ipynb Section 9.
Phases 9-12 should import from here rather than redefining anything.

Design notes:
- GroupNorm is used everywhere a normalization layer is needed (never BatchNorm1d/2d),
  because Phase 9's MC-